In [1]:
import os

# Define the path for the Hospital 2 directory
h2_dir = '../data/contracts/hospital_2/'

# Print the names of the files present in the directory
print("📂 Files found in Hospital 2 directory:")
files = os.listdir(h2_dir)
for f in files:
    print(f"- {f}")

print("\n" + "="*50 + "\n")

# Attempt to read the first text or markdown file found
for f in files:
    if f.endswith('.md') or f.endswith('.txt'):
        file_path = os.path.join(h2_dir, f)
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            print(f"📄 Quick preview of content ({f}):")
            print("-" * 30)
            print(content[:600]) # Print the first 600 characters
            break

📂 Files found in Hospital 2 directory:
- contract_rules.json
- master_services_agreement.md
- master_services_agreement.pdf
- master_services_agreement.txt
- master_services_agreement_scanned.pdf
- service_mapping.json


📄 Quick preview of content (master_services_agreement.md):
------------------------------
# Master Services and Reimbursement Agreement

_Executed as a deed. No tables are used in this instrument._

**Contract number:** INS-H2-2024-1183
**Provider:** St. Auben Metropolitan Hospital Trust
**Payer:** Meridian Health Assurance Group
**Effective from:** 1 January 2024
**Effective to:** 31 December 2025
**Currency:** GBP
**Rounding convention:** half_up_cent

## Article I — Recitals and Term

1.1 This Agreement is made between Meridian Health Assurance Group (the "Payer") and St. Auben Metropolitan Hospital Trust (the "Provider").

1.2 This Agreement takes effect on 1 January 2024 and, 


In [2]:
"""
Script Function: Extract Hospital 2 Contract Rules via Gemini API
Reads the markdown contract, prompts the LLM to extract pricing and rules,
and saves the structured output as a JSON file.
"""
import os
import json
import google.generativeai as genai
from dotenv import load_dotenv

# 1. Load environment variables
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)

# 2. Initialize the Gemini model (using Flash-lite as you previously configured)
model = genai.GenerativeModel('gemini-3.1-flash-lite')

# 3. Read the contract content
contract_path = '../data/contracts/hospital_2/master_services_agreement.md'
with open(contract_path, 'r', encoding='utf-8') as f:
    hospital_2_contract = f.read()

# 4. Define the Prompt (Adapted for text-heavy contracts without tables)
prompt = """
You are an expert medical billing auditor. I will provide you with a hospital provider services agreement.
This contract is written in pure text format with no tables.
Your task is to extract the pricing schedule and billing rules into a structured JSON format.

Extract the following details:
1. contract_metadata: provider_name, contract_number, effective_dates (start and end), currency.
2. services: a list of services where each service has:
   - service_name: The official name of the service in the contract.
   - unit_basis: The basis for billing (e.g., per_procedure, per_hour, per_visit, per_diem).
   - unit_price_cents: The price in cents (integer). If the price in the contract is in major currency (e.g., 150.00), convert it to cents (15000).
   - conditions: Any special conditions, limits, thresholds, or discounts mentioned for this specific service (leave as an empty string if none).

Return ONLY a valid JSON object. Do not include markdown blocks like ```json ... ```.

Here is the contract:
"""

# 5. Save the prompt to a file for documentation
os.makedirs('../prompts', exist_ok=True)
with open('../prompts/prompt_h2_v1.txt', 'w', encoding='utf-8') as f:
    f.write(prompt)

print("⏳ Sending Hospital 2 contract to Gemini API... Please wait.")

# 6. Call the API
response = model.generate_content(prompt + "\n\n" + hospital_2_contract)

# 7. Clean and parse the response
extracted_text = response.text.strip()
if extracted_text.startswith("```json"):
    extracted_text = extracted_text[7:-3].strip()
elif extracted_text.startswith("```"):
    extracted_text = extracted_text[3:-3].strip()

# 8. Save the parsed JSON
try:
    contract_rules = json.loads(extracted_text)
    
    output_path = '../data/contracts/hospital_2/contract_rules.json'
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(contract_rules, f, indent=4)
        
    print(f"✅ Contract rules successfully extracted and saved to {output_path}")
    print("\nPreview of the extracted services:")
    for service in contract_rules.get("services", [])[:3]:
        print(f"- {service['service_name']}: {service['unit_price_cents']} cents ({service['unit_basis']})")

except json.JSONDecodeError:
    print("❌ Failed to parse JSON. Raw output:")
    print(extracted_text)

C:\Users\Nada2\AppData\Local\Temp\ipykernel_23060\288546958.py:8: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


⏳ Sending Hospital 2 contract to Gemini API... Please wait.
✅ Contract rules successfully extracted and saved to ../data/contracts/hospital_2/contract_rules.json

Preview of the extracted services:
- Advanced Infectious Isolation Room Occupancy: 164825 cents (per_diem)
- Assisted Infectious Telemetry Monitoring: 10625 cents (per_hour)
- Comprehensive Otolaryngologic Case Conference: 16525 cents (per_visit)


In [3]:
"""
Script Function: Item Mapping (Fuzzy Matching) for Hospital 2
Matches the raw billing descriptions from Hospital 2 invoices to the official contract terms.
Saves the output to 'service_mapping.json'.
"""

import pandas as pd
import json
from rapidfuzz import process, fuzz

# 1. Setup relative paths for Hospital 2
line_items_path = '../data/invoices/hospital_2_line_items.csv'
contract_rules_path = '../data/contracts/hospital_2/contract_rules.json'
mapping_output_path = '../data/contracts/hospital_2/service_mapping.json'

# 2. Load data
line_items_df = pd.read_csv(line_items_path)
with open(contract_rules_path, 'r', encoding='utf-8') as f:
    contract_rules = json.load(f)

# 3. Extract official service names and billed descriptions
official_services = [service['service_name'] for service in contract_rules['services']]
billed_descriptions = line_items_df['description'].unique()

print("⏳ Mapping billed descriptions to official contract services for Hospital 2...")

# 4. Perform Fuzzy Matching
mapping_dict = {}
for billed_desc in billed_descriptions:
    # Clean description (remove specific codes if present)
    clean_desc = billed_desc.split('/')[0].strip() if '/' in billed_desc else billed_desc
    
    # Find the best match using RapidFuzz
    best_match = process.extractOne(clean_desc, official_services, scorer=fuzz.token_sort_ratio)
    
    if best_match:
        matched_service, score, _ = best_match
        mapping_dict[billed_desc] = {
            "mapped_service": matched_service,
            "confidence_score": round(score, 2)
        }

# 5. Save the mapping to a JSON file
with open(mapping_output_path, 'w', encoding='utf-8') as f:
    json.dump(mapping_dict, f, indent=4)

print(f"✅ Mapping complete for Hospital 2! Saved to: {mapping_output_path}")

⏳ Mapping billed descriptions to official contract services for Hospital 2...
✅ Mapping complete for Hospital 2! Saved to: ../data/contracts/hospital_2/service_mapping.json


In [ ]:
"""
Script Function: Invoice Auditor Engine & Submission Generator (Hospital 2)
Applies the contract rules to the billed line items for Hospital 2.
Calculates the 'expected' price, compares it with the billed total, flags errors,
and generates the final 'submission.csv' file formatted for delivery.
"""

import pandas as pd
import json
import re

# 1. Setup relative paths for Hospital 2
invoices_path = '../data/invoices/hospital_2_invoices.csv'
line_items_path = '../data/invoices/hospital_2_line_items.csv'
contract_rules_path = '../data/contracts/hospital_2/contract_rules.json'
mapping_path = '../data/contracts/hospital_2/service_mapping.json'

# 2. Load data
invoices_df = pd.read_csv(invoices_path)
line_items_df = pd.read_csv(line_items_path)

with open(contract_rules_path, 'r', encoding='utf-8') as f:
    contract_rules = json.load(f)

with open(mapping_path, 'r', encoding='utf-8') as f:
    service_mapping = json.load(f)

rules_dict = {item['service_name']: item for item in contract_rules['services']}

# 3. Pricing Logic Function
def calculate_expected_line_total(quantity, base_price, condition_str):
    if not condition_str or pd.isna(condition_str):
        return quantity * base_price
    
    # Thresholds 
    threshold_match = re.search(r'Threshold premium:\s*\+(\d+)%\s*if daily qty >\s*(\d+)', str(condition_str), re.IGNORECASE)
    if threshold_match:
        premium_percent = int(threshold_match.group(1))
        threshold_qty = int(threshold_match.group(2))
        if quantity > threshold_qty:
            base_units = threshold_qty
            premium_units = quantity - threshold_qty
            premium_price = round(base_price * (1 + (premium_percent / 100.0)))
            return (base_units * base_price) + (premium_units * premium_price)
        else:
            return quantity * base_price

    # Caps
    cap_match = re.search(r'(?:daily\s+cap|max.*?quantity).*?(?<!\$)\b(\d+)\b(?!%)', str(condition_str), re.IGNORECASE)
    if cap_match:
        cap_qty = int(cap_match.group(1))
        allowed_qty = min(quantity, cap_qty)
        return allowed_qty * base_price
        
    # Discounts 
    discount_match = re.search(r'(\d+)%\s+discount.*?qty\s*>\s*(\d+)', str(condition_str), re.IGNORECASE)
    if discount_match:
        disc_percent = int(discount_match.group(1))
        disc_qty = int(discount_match.group(2))
        if quantity > disc_qty:
            discounted_price = round(base_price * (1 - (disc_percent / 100.0)))
            return quantity * discounted_price
            
    return quantity * base_price

# 4. Apply Logic to line items
print("⏳ Auditing Hospital 2 line items...")
expected_totals = []
error_cats = []
confidences = []

for idx, row in line_items_df.iterrows():
    billed_desc = row['description']
    quantity = row['quantity']
    
    mapping_data = service_mapping.get(billed_desc, {})
    official_service = mapping_data.get('mapped_service')
    confidence = mapping_data.get('confidence_score', 0)
    
    if official_service and official_service in rules_dict and confidence >= 60:
        base_price = rules_dict[official_service]['unit_price_cents']
        condition_str = rules_dict[official_service]['conditions']
        expected_total = calculate_expected_line_total(quantity, base_price, condition_str)
        error_cat = "Clean"
    else:
        expected_total = 0
        error_cat = "Unmapped Service"
        confidence = 0.0
        
    expected_totals.append(expected_total)
    error_cats.append(error_cat)
    confidences.append(confidence / 100.0)

line_items_df['expected_line_total'] = expected_totals
line_items_df['line_error_category'] = error_cats
line_items_df['line_confidence'] = confidences

# 5. Aggregate to Invoice Level
print("⏳ Aggregating to invoice level...")
invoice_summary = line_items_df.groupby('invoice_id').agg(
    expected_total_cents=('expected_line_total', 'sum'),
    min_confidence=('line_confidence', 'min')
).reset_index()

final_audit_df = invoices_df.merge(invoice_summary, on='invoice_id', how='left')

# 6. Flag Errors
#Handle NaNs
final_audit_df['expected_total_cents'] = final_audit_df['expected_total_cents'].fillna(0)
final_audit_df['min_confidence'] = final_audit_df['min_confidence'].fillna(0.0)
# Add tolerance of 2 cents
final_audit_df['flagged'] = (abs(final_audit_df['invoice_total_cents'] - final_audit_df['expected_total_cents']) > 2).astype(int)
def categorize_error(row):
    if row['flagged'] == 0:
        return ""
    if row['min_confidence'] < 0.6:
        return "Low Confidence Mapping"
    if row['invoice_total_cents'] > row['expected_total_cents']:
        return "Overbilled"
    return "Underbilled"

final_audit_df['error_category'] = final_audit_df.apply(categorize_error, axis=1)

# 7. Generate Final Submission File
print("⏳ Generating final submission.csv...")
submission_df = pd.DataFrame({
    'invoice_id': final_audit_df['invoice_id'],
    'flagged': final_audit_df['flagged'],
    'error_category': final_audit_df['error_category'],
    'expected_total_cents': final_audit_df['expected_total_cents'].astype(int),
    'billed_total_cents': final_audit_df['invoice_total_cents'].astype(int),
    'confidence': final_audit_df['min_confidence'].round(2)
})

submission_path = '../submission.csv'
submission_df.to_csv(submission_path, index=False)

print(f"✅ Audit complete! Final submission file generated at: {submission_path}")
print("\nPreview of flagged invoices for Hospital 2:")
print(submission_df[submission_df['flagged'] == 1].head())

⏳ Auditing Hospital 2 line items...
⏳ Aggregating to invoice level...
⏳ Generating final submission.csv...
✅ Audit complete! Final submission file generated at: ../submission.csv

Preview of flagged invoices for Hospital 2:
      invoice_id  flagged          error_category  expected_total_cents  \
0  INV-H2-000001        1  Low Confidence Mapping               1367425   
1  INV-H2-000002        1  Low Confidence Mapping               2600700   
2  INV-H2-000003        1  Low Confidence Mapping               1413725   
3  INV-H2-000004        1  Low Confidence Mapping               2742825   
4  INV-H2-000005        1  Low Confidence Mapping               3936700   

   billed_total_cents  confidence  
0             1562227        0.20  
1             3209350        0.19  
2             2215650        0.19  
3             3434580        0.20  
4             1887000        0.22  
